# Validacion de integridad de los datos

In [ ]:
def auditar_integridad_y_esquema(ruta_archivo):
    """
    Realiza una auditoría completa del archivo crudo verificando su integridad
    criptográfica (Checksum SHA-256) y validando su esquema estructural básico.

    UTILIDAD PARA EL NEGOCIO Y JUSTIFICACIÓN TÉCNICA:
    1. Verificación de Integridad (Checksum): En un entorno profesional, los datos
       pueden corromperse durante la extracción o transferencia. El hash SHA-256
       actúa como una "huella digital" del archivo. Si un solo byte cambia en el
       origen, el hash será completamente distinto, alertando al equipo de Data
       Engineering antes de que datos corruptos entren al modelo.

    2. Validación de Esquema: Asegura de forma temprana que el archivo contiene
       las variables esperadas (nombres de columnas) antes de invertir recursos
       computacionales en el Pipeline de transformación.

    Parámetros:
    ruta_archivo (str): La ruta del archivo CSV a auditar.

    Retorna:
    str: El hash SHA-256 del archivo si es exitoso.
    """
    print("Iniciando Auditoría de Integridad y Esquema")

    # Inicializamos el objeto que calculará el hash SHA-256.
    sha256_hash = hashlib.sha256()

    try:
        # Abrimos el archivo en modo "rb" (read binary). Es necesario leerlo
        # en binario (como ceros y unos) porque los algoritmos criptográficos
        # como SHA-256 operan a nivel de bytes, no de texto.
        with open(ruta_archivo, "rb") as f:

            # BUCLE DE LECTURA OPTIMIZADA:
            # En lugar de cargar todo el archivo a la memoria (lo que podría
            # colapsar la RAM si el CSV pesa varios GBs), usamos iter() con un lambda.
            # lambda: f.read(4096) -> Lee el archivo en "trozos" (chunks) de 4096 bytes (4KB).

            # b"" -> Es el "centinela" (sentinel). Le dice al iterador: "sigue leyendo
            # trozos de 4KB hasta que el archivo te devuelva un string binario vacío (b""),
            # lo que significa que llegaste al final del archivo".
            for byte_block in iter(lambda: f.read(4096), b""):
                # Por cada trozo de 4KB leído, actualizamos la calculadora de hash.
                sha256_hash.update(byte_block)

        # Una vez procesado todo el archivo, hexdigest() convierte el resultado
        # binario final en una cadena de texto legible (hexadecimal). Esta es
        # nuestra "huella digital" definitiva.
        checksum = sha256_hash.hexdigest()

        # Mostramos mensajes de éxito en consola informando el archivo procesado y su Hash.
        print("INTEGRIDAD: Checksum SHA-256 calculado correctamente.")
        print(f"   -> Archivo: {ruta_archivo}")
        print(f"   -> Hash: {checksum}")

    # Manejo de error específico: Si la ruta que le dimos a la función no existe,
    # capturamos el error y detenemos la función retornando 'None' para evitar
    # que el script completo se rompa.
    except FileNotFoundError:
        print(f"ERROR: No se encontró el archivo en la ruta: {ruta_archivo}")
        return None

    try:
        # Utilizamos pd.read_csv pero con el argumento clave 'nrows=0'.
        # Esto le ordena a Pandas: "Lee solamente la primera fila del CSV (donde
        # están los nombres de las columnas) y construye un DataFrame vacío sin
        # procesar ni un solo registro de datos". Es un escaneo en milisegundos.
        df_esquema = pd.read_csv(ruta_archivo, nrows=0)

        # Extraemos los nombres de esas columnas en una lista estándar de Python.
        columnas_encontradas = df_esquema.columns.tolist()

        # Definimos nuestra lista. Estas son las columnas
        # que el negocio ha definido como obligatorias para que el modelo predictivo
        # (tu Pipeline) pueda funcionar correctamente sin caerse.
        columnas_criticas = ['id_cliente', 'abandono', 'ingreso_mensual', 'deuda_total']

        # COMPRESIÓN DE LISTAS:
        # Es un filtro rápido. Leemos la lista de columnas críticas y preguntamos:
        # "Para cada columna crítica, ¿NO está (not in) en la lista de columnas
        # encontradas en el archivo?". Guardamos las que faltan en una nueva lista.
        columnas_faltantes = [col for col in columnas_criticas if col not in columnas_encontradas]


        # Si la lista 'columnas_faltantes' está vacía (not columnas_faltantes),
        # significa que todas pasaron la prueba de asistencia.
        if not columnas_faltantes:
            print("ESQUEMA: Todas las variables críticas están presentes en el origen.")
        # Si la lista contiene elementos, imprimimos una advertencia clara diciendo
        # exactamente qué variables no vinieron en el archivo.
        else:
            print(f"ADVERTENCIA: Faltan las siguientes columnas críticas: {columnas_faltantes}")

    # Manejo de error genérico (Exception) para atrapar cualquier otro problema
    # inesperado al leer el archivo (ej. permisos denegados, formato corrupto).
    except Exception as e:
        print(f"ERROR al validar el esquema: {e}")
        return None

    # Finalmente, si ambas validaciones pasaron (el archivo existe, se pudo
    # hacer el hash y se leyó el esquema), la función devuelve la huella digital.
    return checksum

# Invocamos la función pasando el nombre exacto de nuestro archivo crudo.
# El resultado (el hash de seguridad) se guarda en la variable 'hash_dataset'
# para ser usado como comprobante en MLOps (Data Science).
hash_dataset = auditar_integridad_y_esquema('dataset_clientes.csv')

Iniciando Auditoría de Integridad y Esquema
INTEGRIDAD: Checksum SHA-256 calculado correctamente.
   -> Archivo: dataset_clientes.csv
   -> Hash: ac8ed445faf148dac481236a2c045716ef0a6d6f54f6f8fbab6d8c5efc7ecfac
ESQUEMA: Todas las variables críticas están presentes en el origen.
